In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split


from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE


import warnings
warnings.filterwarnings('ignore')

In [65]:
import pandas as pd
import numpy as np

# Load Data
df = pd.read_csv('marketing_campaign.csv', sep='\t')

# Data Inspection
print("=== Data Info ===")
df.info()

# Duplicate Detection
duplicates = df.duplicated().sum()
print(f"\nNumber of exact duplicates: {duplicates}")

# Missing Values
print("\n=== Missing Values Check ===")
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0])


=== Data Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases   

In [66]:
df.head(1)

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1


In [67]:
df.value_counts()

,,,,,,,,,,,,,,,,,,,,,,,,,,,,,count
ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response,
11191,1986,Graduation,Divorced,41411.0,0,0,07-12-2013,11,37,32,38,11,3,18,1,2,1,4,6,0,0,0,0,0,0,3,11,0,1
0,1985,Graduation,Married,70951.0,0,0,04-05-2013,66,239,10,554,254,87,54,1,3,4,9,1,0,0,0,0,0,0,3,11,0,1
1,1961,Graduation,Single,57091.0,0,0,15-06-2014,0,464,5,64,7,0,37,1,7,3,7,5,0,0,0,0,1,0,3,11,1,1
9,1975,Master,Single,46098.0,1,1,18-08-2012,86,57,0,27,0,0,36,4,3,2,2,8,0,0,0,0,0,0,3,11,0,1
13,1947,PhD,Widow,25358.0,0,1,22-07-2013,57,19,0,5,0,0,8,2,1,0,3,6,0,0,0,0,0,0,3,11,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55,1963,Graduation,Together,56253.0,0,1,07-12-2012,83,509,0,65,7,11,5,4,7,2,9,6,0,0,0,0,0,0,3,11,0,1
49,1970,Graduation,Single,20587.0,1,0,11-05-2014,39,2,3,6,4,1,9,1,1,1,2,7,0,0,0,0,0,0,3,11,0,1
48,1964,Graduation,Together,55761.0,0,1,24-04-2014,97,136,1,12,0,3,32,2,4,1,3,6,0,1,0,0,0,0,3,11,0,1


In [68]:
df.shape

(2240, 29)

In [69]:
counts = df[['Z_CostContact', 'Z_Revenue']].apply(pd.value_counts)
print(counts)

    Z_CostContact  Z_Revenue
3          2240.0        NaN
11            NaN     2240.0


In [70]:
df['Age'] = 2026 - df['Year_Birth']
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
max_date = df['Dt_Customer'].max()

# معانا بقاله قد ايه
df['Customer_Days'] = (max_date - df['Dt_Customer']).dt.days

df['Total_Spent'] = df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] + \
                    df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds']


df['Total_Children'] = df['Kidhome'] + df['Teenhome']


cols_to_drop = ['ID', 'Year_Birth', 'Dt_Customer', 'Z_CostContact', 'Z_Revenue']
df = df.drop(columns=cols_to_drop)


df = df[(df['Age'] < 100) & ((df['Income'] < 600000) | df['Income'].isnull())]

print("Feature Engineering & Outlier Removal Completed!")
print(f"Data shape after cleaning: {df.shape}")

Feature Engineering & Outlier Removal Completed!
Data shape after cleaning: (2236, 28)


In [81]:
# Column Separation
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns

# Numerical Pipeline
num_pipeline = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())  ])

# Categorical Pipeline
cat_pipeline = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))         ])

# ColumnTransformer
#preprocessor = ColumnTransformer(transformers=[('num', num_pipeline, num_cols),#('cat', cat_pipeline, cat_cols) ], remainder='drop')

#X_processed = preprocessor.fit_transform(df)

print(f"Original shape: {df.shape}")
print(f"Processed shape: {X_processed.shape}")

Original shape: (2236, 28)
Processed shape: (2236, 39)


In [82]:
df.head(1)

,Education,Marital_Status,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,...,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,Age,Customer_Days,Total_Spent,Total_Children
0,Graduation,Single,58138.0,0,0,58,635,88,546,172,...,0,0,0,0,0,1,69,663,1617,0


In [83]:
# Train/Test Split
X = df.drop(columns=['Response'])
y = df['Response']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical columns found: {len(num_cols)}")
print(f"Categorical columns found: {len(cat_cols)}")


# preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
], remainder='drop')

# The Master Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
final_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('selector', SelectKBest(score_func=f_classif, k=10)),
    ('classifier', RandomForestClassifier(random_state=42))
])

final_pipeline.fit(X_train, y_train)

accuracy = final_pipeline.score(X_test, y_test)
print(f"Final Pipeline Accuracy: {accuracy * 100:.2f}%")

Numerical columns found: 25
Categorical columns found: 2
Final Pipeline Accuracy: 84.82%
